# Indexing TREC Robust 2005 by OpenSearch for Dense Encoder Model

Chunks and dense-encodes each passage entirely server-side using the remotely
hosted `intfloat/multilingual-e5-large` model (passage encoder, model id
`31w-Kp8BFwfKFgbhNVZp`) registered in OpenSearch.

- [aquaint/trec-robust-2005](https://ir-datasets.com/aquaint.html#aquaint/trec-robust-2005)
- Prerequisite: [ml_model_registration.ipynb](ml_model_registration.ipynb)

In [1]:
import sys
!{sys.executable} -m pip install -q ir_datasets pandas opensearch-py dotenv beautifulsoup4

In [2]:
import pprint
from tqdm import tqdm

### Create an OpenSearch Client

Your opensearch password should be available in `~/.env`

```bash
    OPENSEARCH_INITIAL_ADMIN_PASSWORD="strong password"
```

In [3]:
import os
from dotenv import load_dotenv
from opensearchpy import OpenSearch

load_dotenv()
host = 'localhost'
port = 9200
password = os.getenv("OPENSEARCH_INITIAL_ADMIN_PASSWORD")

client = OpenSearch(
    hosts=[{"host": host, "port": port}],
    http_auth=("admin", password),
    http_compress=True,
    use_ssl=True,
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)
pprint.pprint(client.info())

{'cluster_name': 'docker-cluster',
 'cluster_uuid': 'qIJ28Ej2TCC6_LI88zZLlw',
 'name': '4db878c40bab',
 'tagline': 'The OpenSearch Project: https://opensearch.org/',
 'version': {'build_date': '2026-02-07T07:54:31.169913465Z',
             'build_hash': 'bbc94f0bdc3a759011e6529ecfe52840856f91a3',
             'build_snapshot': False,
             'build_type': 'tar',
             'distribution': 'opensearch',
             'lucene_version': '10.3.2',
             'minimum_index_compatibility_version': '2.0.0',
             'minimum_wire_compatibility_version': '2.19.0',
             'number': '3.5.0'}}


### Index a Corpus for DPR Model

Encoding runs on the remote model host, so no local GPU / sentence-transformers is needed here.

In [4]:
import ir_datasets
dataset_name = "aquaint/trec-robust-2005"
dataset = ir_datasets.load(dataset_name)

In [5]:
index_name = "trec_robust_2005_dpr"

In [6]:
# Delete an existing index (be careful)
if client.indices.exists(index=index_name):
    response = client.indices.delete(index=index_name)
    pprint.pprint(response)
else:
    print(f"{index_name} does not exist")

trec_robust_2005_dpr does not exist


### Dense Encoding of Chunks (server-side ingest pipeline)

Chunk **and** dense-encode entirely inside OpenSearch. Bulk sends *raw* documents; a single ingest pipeline runs two processors in order:

1. `text_chunking` — splits `text` into passages in `text_chunks`.
2. `text_embedding` — calls the remote `multilingual-e5-large` model on each passage and writes per-chunk `knn_vector`s to the nested `text_chunks_embedding` field.

In [ ]:
pipeline_id = "trec_robust_2005_chunk_dense"
model_id = "your-model-id"  # dense passage encoding model registered in OpenSearch

In [8]:
def create_chunk_dense_pipeline(
    pipeline_id: str,
    model_id: str,
    source_field: str = "text",
    chunk_field: str = "text_chunks",
    embedding_field: str = "text_chunks_embedding",
    token_limit: int = 384,
    overlap_rate: float = 0.2,
    tokenizer: str = "standard",
    batch_size: int = 16,
) -> dict:
    """
    Create (or update) an ingest pipeline that chunks then dense-encodes text,
    fully server-side.

    Stage 1 (`text_chunking`) splits `source_field` into passages in `chunk_field`.
    Stage 2 (`text_embedding`) embeds each passage with the remote `model_id`,
    writing a nested list of knn_vectors to `embedding_field`.

    `batch_size` bundles that many *documents'* chunks into a single model call
    (batch ingestion), so the GPU processes them as one padded batch instead of
    one at a time. Actual texts per call ~= batch_size x avg chunks/doc.
    """
    body = {
        "description": "Chunk documents, then dense-encode each passage",
        "processors": [
            {
                "text_chunking": {
                    "algorithm": {
                        "fixed_token_length": {
                            "token_limit": token_limit,
                            "overlap_rate": overlap_rate,
                            "tokenizer": tokenizer,
                        }
                    },
                    "field_map": {source_field: chunk_field},
                }
            },
            {
                "text_embedding": {
                    "model_id": model_id,
                    "field_map": {chunk_field: embedding_field},
                    "batch_size": batch_size,
                }
            },
        ],
    }
    return client.ingest.put_pipeline(id=pipeline_id, body=body)

response = create_chunk_dense_pipeline(pipeline_id, model_id, batch_size=64)
pprint.pprint(response)

{'acknowledged': True}


Bulk indexing

In [9]:
index_body = {
  "settings": {
    "index": {
      "number_of_shards": 1,
      "number_of_replicas": 0,
      # Enable approximate k-NN so the knn_vector fields build a search graph.
      "knn": True
    }
    # English corpus: rely on OpenSearch's default (standard) analyzer.
  },
  "mappings": {
    "properties": {
        "docid": { "type": "keyword" },
        "title": { "type": "text" },
        "text": { "type": "text" },
        "text_chunks": { "type": "text" },
        # Per-chunk dense vectors produced by the text_embedding processor.
        # Chunking yields a list of passages, so the embeddings must be `nested`.
        # multilingual-e5-large emits 1024-dim vectors tuned for cosine similarity.
        "text_chunks_embedding": {
            "type": "nested",
            "properties": {
                "knn": {
                    "type": "knn_vector",
                    "dimension": 1024,
                    "space_type": "cosinesimil"
                }
            }
        },
    }
  }
}

response = client.indices.create(index=index_name, body=index_body)
pprint.pprint(response)

{'acknowledged': True,
 'index': 'trec_robust_2005_dpr',
 'shards_acknowledged': True}


In [10]:
from bs4 import BeautifulSoup
def parse_marked_up_doc(marked_up_doc):
    # Parse the content using BeautifulSoup
    soup = BeautifulSoup(marked_up_doc, 'html.parser')

    # Extract the title from the <HEADLINE> tag (empty string if absent)
    headline_tag = soup.find('headline')
    title = headline_tag.get_text(strip=True) if headline_tag else ""

    # Extract the body text by joining the <P> paragraph tags
    text = ' '.join(p.get_text(strip=True) for p in soup.find_all('p'))

    return title, text

In [11]:
# Cap doc length so a single very long doc can't explode into hundreds of chunks
# (each chunk is dense-encoded on the model host; unbounded chunks/doc OOMs it).
MAX_DOC_CHARS = 10_000

def prepare_documents(dataset):
    """
    Yield raw bulk actions. Chunking + encoding happen server-side in the
    ingest pipeline, so we only ship docid/title/text.

    AQUAINT wraps the title in <HEADLINE> and the body in <P> tags, so we
    parse `marked_up_doc` to keep the title as a separate field. Only `text`
    is chunked/encoded, so an empty title (docs with no <HEADLINE>) has no
    effect on the embeddings.
    """
    for doc in dataset.docs_iter():
        title, text = parse_marked_up_doc(doc.marked_up_doc)
        text = text.replace("\n", " ")[:MAX_DOC_CHARS]
        yield {
            "_id": doc.doc_id,  # Unique identifier for the document
            "_source": {
                "docid": doc.doc_id,
                "title": title,
                "text": text
            }
        }

In [12]:
from opensearchpy.helpers import streaming_bulk

# Batch inference is configured server-side in the pipeline's `text_embedding`
# processor (batch_size=64). OpenSearch 3.x removed the `_bulk?batch_size=` query
# param (2.x only), so we must NOT pass one here — it 400s the whole request.

# Total for the progress bar (docs_count is instant; fall back to a full id scan).
try:
    total = dataset.docs_count()
except Exception:
    total = sum(1 for _ in dataset.docs_iter())

success, errors = 0, []
with tqdm(total=total, desc="Indexing") as bar:
    for ok, item in streaming_bulk(
        client,
        prepare_documents(dataset),
        index=index_name,
        pipeline=pipeline_id,
        chunk_size=128,                # 2 x processor batch_size; bar advances every 128 docs
        request_timeout=300,
        max_retries=3,
        initial_backoff=2,
        raise_on_error=False,          # collect failures instead of aborting the run
        raise_on_exception=False,
    ):
        bar.update(1)                  # advances per actually-processed doc
        success += ok
        if not ok:
            errors.append(item)

print(f"indexed: {success},  failed: {len(errors)}")
if errors:
    pprint.pprint(errors[:3])          # inspect the first few errors

Indexing: 100%|██████████| 1033461/1033461 [5:33:35<00:00, 51.63it/s]  

indexed: 1033461,  failed: 0
